# finqa-lora: GPU job runner

Thin wrapper. All logic lives in the repo; this notebook clones it at `REF`, installs the pinned
libraries on top of Kaggle's torch, downloads the pinned FinQA data and runs one job. Outputs go to
`/kaggle/working/<job name>/` and are saved with the notebook version.

Settings before **Save Version → Save & Run All (Commit)**: Accelerator **GPU T4 x2** (or T4), Internet **on**.

In [ ]:
REPO = "https://github.com/PritRai75/finqa-lora.git"
REF = "main"                 # a branch or a commit hash
JOB_NAME = "feasibility"
JOB = f"python -m finqa_lora.feasibility --out /kaggle/working/{JOB_NAME}"
WORKDIR = "/tmp/finqa-lora"  # code and data stay out of the saved outputs

In [ ]:
import sys, torch
print(sys.version)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| gpus", torch.cuda.device_count())
assert torch.cuda.is_available(), "No GPU. Settings > Accelerator > GPU T4 x2, then save again."
major, minor = (int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
assert (major, minor) >= (2, 6), "This torch is too old for TRL 1.17 (needs 2.6+)."
!nvidia-smi

In [ ]:
!rm -rf {WORKDIR} && git clone --quiet {REPO} {WORKDIR} && cd {WORKDIR} && git checkout --quiet {REF} && git log -1 --format="commit %H  %s"
!pip install -q -r {WORKDIR}/requirements-gpu.txt
!cd {WORKDIR} && python -m finqa_lora.fetch

In [ ]:
# One GPU only: with two visible, the Hugging Face Trainer would switch to DataParallel.
!cd {WORKDIR} && CUDA_VISIBLE_DEVICES=0 {JOB} 2>&1 | tee /kaggle/working/{JOB_NAME}.log
!pip freeze > /kaggle/working/{JOB_NAME}-pip-freeze.txt

In [ ]:
import json, pathlib
p = pathlib.Path(f"/kaggle/working/{JOB_NAME}/report.json")
print(p.read_text() if p.exists() else "no report.json: see the log above")